In [ ]:
# ============================================================
# Replication code for main-paper Tables 1 and 2
#
# Table 1: KS diagnostics for nearly stationary statistic T_n
#          Target law: N(0, 2c)
#
# Table 2: KS diagnostics for mildly explosive statistic S_n
#          Target law: standard Cauchy
#
# Designs:
#   (i) Homoskedastic: sigma_t = 1
#   (ii) Stochastic volatility:
#        log(sigma_t^2) = phi_n log(sigma_{t-1}^2) + eta_t,
#        phi_n = 1 - d/log(log n)
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import kstest

# -----------------------------
# Global settings
# -----------------------------

np.random.seed(123)

N = 10_000
C = 2.0
D = 1.0
ALPHA = 0.1

OUTER_REPS = 100
INNER_MC = 100
TOTAL_MC = OUTER_REPS * INNER_MC
BATCH_SIZE = 500

# -----------------------------
# k_n designs
# -----------------------------

# Table 1: nearly stationary case, includes fixed k_n = 10
K_SPECS_NEAR = {
    r"$10$": lambda n: 10.0,
    r"$\log n$": lambda n: np.log(n),
    r"$n^{0.10}$": lambda n: n**0.10,
    r"$n^{0.25}$": lambda n: n**0.25,
    r"$n^{0.50}$": lambda n: n**0.50,
    r"$n^{0.75}$": lambda n: n**0.75,
    r"$n/\log n$": lambda n: n / np.log(n),
}

# Table 2: mildly explosive case, includes n^0.90
K_SPECS_EXPLOSIVE = {
    r"$\log n$": lambda n: np.log(n),
    r"$n^{0.10}$": lambda n: n**0.10,
    r"$n^{0.25}$": lambda n: n**0.25,
    r"$n^{0.50}$": lambda n: n**0.50,
    r"$n^{0.75}$": lambda n: n**0.75,
    r"$n^{0.90}$": lambda n: n**0.90,
    r"$n/\log n$": lambda n: n / np.log(n),
}


# ============================================================
# Simulation functions
# ============================================================

def simulate_homoskedastic_stats(n, k, c, regime):
    """
    Simulate TOTAL_MC normalized OLS statistics under homoskedastic errors.

    Near-stationary statistic:
        T_n = sqrt(n k_n) (rho_hat - rho_n)

    Mildly explosive statistic:
        S_n = rho_n^n k_n (rho_hat - rho_n) / (2c)

    The mildly explosive statistic is computed using a scaled recursion to avoid
    numerical overflow from rho_n^n.
    """

    all_stats = []
    remaining = TOTAL_MC

    while remaining > 0:
        m = min(BATCH_SIZE, remaining)
        remaining -= m

        if regime == "near":
            rho = 1.0 - c / k

            y_prev = np.zeros(m)
            num = np.zeros(m)
            den = np.zeros(m)

            for _ in range(n):
                u = np.random.randn(m)

                num += y_prev * u
                den += y_prev**2

                y_prev = rho * y_prev + u

            stats = np.sqrt(n * k) * (num / den)

        elif regime == "explosive":
            rho = 1.0 + c / k
            log_rho = np.log1p(c / k)

            # scaled state: x_t = rho^{-t} y_t
            x_prev = np.zeros(m)
            num_scaled = np.zeros(m)
            den_scaled = np.zeros(m)

            for t in range(1, n + 1):
                u = np.random.randn(m)
                w = np.exp(-(n - t + 1) * log_rho)

                num_scaled += w * x_prev * u
                den_scaled += (w**2) * (x_prev**2)

                x_prev += np.exp(-t * log_rho) * u

            stats = (k / (2.0 * c)) * (num_scaled / den_scaled)

        else:
            raise ValueError("regime must be 'near' or 'explosive'")

        all_stats.append(stats)

    stats = np.concatenate(all_stats)
    return stats.reshape(OUTER_REPS, INNER_MC)


def simulate_sv_stats(n, k, c, d, alpha, regime):
    """
    Simulate TOTAL_MC normalized OLS statistics under stochastic volatility.

    Volatility model:
        log(sigma_t^2) = phi_n log(sigma_{t-1}^2) + eta_t,
        phi_n = 1 - d/log(log n),
        eta_t ~ N(0, alpha^2).
    """

    phi = 1.0 - d / np.log(np.log(n))

    all_stats = []
    remaining = TOTAL_MC

    while remaining > 0:
        m = min(BATCH_SIZE, remaining)
        remaining -= m

        z = np.zeros(m)

        if regime == "near":
            rho = 1.0 - c / k

            y_prev = np.zeros(m)
            num = np.zeros(m)
            den = np.zeros(m)

            for _ in range(n):
                eps = np.random.randn(m)
                eta = alpha * np.random.randn(m)

                z = phi * z + eta
                u = np.exp(0.5 * z) * eps

                num += y_prev * u
                den += y_prev**2

                y_prev = rho * y_prev + u

            stats = np.sqrt(n * k) * (num / den)

        elif regime == "explosive":
            rho = 1.0 + c / k
            log_rho = np.log1p(c / k)

            # scaled state: x_t = rho^{-t} y_t
            x_prev = np.zeros(m)
            num_scaled = np.zeros(m)
            den_scaled = np.zeros(m)

            for t in range(1, n + 1):
                eps = np.random.randn(m)
                eta = alpha * np.random.randn(m)

                z = phi * z + eta
                u = np.exp(0.5 * z) * eps

                w = np.exp(-(n - t + 1) * log_rho)

                num_scaled += w * x_prev * u
                den_scaled += (w**2) * (x_prev**2)

                x_prev += np.exp(-t * log_rho) * u

            stats = (k / (2.0 * c)) * (num_scaled / den_scaled)

        else:
            raise ValueError("regime must be 'near' or 'explosive'")

        all_stats.append(stats)

    stats = np.concatenate(all_stats)
    return stats.reshape(OUTER_REPS, INNER_MC)


# ============================================================
# KS diagnostics
# ============================================================

def ks_summary(stats_matrix, regime):
    """
    Run one KS test for each outer replication.

    Returns:
        average KS statistic,
        acceptance proportion = fraction of p-values above 0.05.
    """

    ks_vals = []
    accept = []

    for b in range(OUTER_REPS):
        x = stats_matrix[b, :]
        x = x[np.isfinite(x)]

        if len(x) < 2:
            ks_vals.append(np.nan)
            accept.append(np.nan)
            continue

        if regime == "near":
            result = kstest(x, "norm", args=(0.0, np.sqrt(2.0 * C)))
        elif regime == "explosive":
            result = kstest(x, "cauchy", args=(0.0, 1.0))
        else:
            raise ValueError("regime must be 'near' or 'explosive'")

        ks_vals.append(result.statistic)
        accept.append(result.pvalue > 0.05)

    return np.nanmean(ks_vals), np.nanmean(accept)


def run_one_design(k_name, k_value, regime, model):
    """
    Run one k_n design for either the homoskedastic or SV model.
    """

    if model == "homo":
        stats_matrix = simulate_homoskedastic_stats(
            n=N,
            k=k_value,
            c=C,
            regime=regime
        )

    elif model == "sv":
        stats_matrix = simulate_sv_stats(
            n=N,
            k=k_value,
            c=C,
            d=D,
            alpha=ALPHA,
            regime=regime
        )

    else:
        raise ValueError("model must be 'homo' or 'sv'")

    mean_ks, acc_prop = ks_summary(stats_matrix, regime)

    return {
        "k_n": k_name,
        "Mean KS stat": mean_ks,
        "Acceptance Proportion": acc_prop
    }


def run_table(k_specs, regime):
    """
    Run all k_n designs and combine homoskedastic and SV outputs.
    """

    homo_rows = []
    sv_rows = []

    for k_name, k_fun in k_specs.items():
        k_value = float(k_fun(N))

        print(f"Running homoskedastic | regime = {regime:10s} | k_n = {k_name}")
        homo_rows.append(run_one_design(k_name, k_value, regime, model="homo"))

        print(f"Running SV           | regime = {regime:10s} | k_n = {k_name}")
        sv_rows.append(run_one_design(k_name, k_value, regime, model="sv"))

    homo_df = pd.DataFrame(homo_rows)
    sv_df = pd.DataFrame(sv_rows)

    merged = homo_df.merge(
        sv_df,
        on="k_n",
        suffixes=("_Homo", "_SV")
    )

    return merged


# ============================================================
# Run main-paper tables
# ============================================================

# Main-paper Table 1
table1_near_stationary = run_table(K_SPECS_NEAR, regime="near")

# Main-paper Table 2
table2_mild_explosive = run_table(K_SPECS_EXPLOSIVE, regime="explosive")

print("\n==============================")
print("Main-paper Table 1: Nearly stationary KS diagnostics")
print("==============================")
print(table1_near_stationary.round(4))

print("\n==============================")
print("Main-paper Table 2: Mildly explosive KS diagnostics")
print("==============================")
print(table2_mild_explosive.round(4))


# ============================================================
# Save CSV files
# ============================================================

table1_near_stationary.to_csv("table1_near_stationary_ks.csv", index=False)
table2_mild_explosive.to_csv("table2_mild_explosive_ks.csv", index=False)


# ============================================================
# LaTeX table output
# ============================================================

def print_latex_combined_table(df, caption, label):
    """
    Print a LaTeX table matching the main-paper format:
    Homoskedastic columns + Stochastic-volatility columns.
    """

    print("\n" + caption)
    print(r"\begin{table}[H]")
    print(r"\centering")
    print(r"\begin{tabular}{lrrrr}")
    print(r"\toprule")
    print(r"& \multicolumn{2}{c}{Homoskedastic} & \multicolumn{2}{c}{Stochastic volatility} \\")
    print(r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}")
    print(r"$k_n$ & Mean KS stat & Acceptance prop. & Mean KS stat & Acceptance prop. \\")
    print(r"\midrule")

    for _, row in df.iterrows():
        print(
            f"{row['k_n']} & "
            f"{row['Mean KS stat_Homo']:.4f} & "
            f"{row['Acceptance Proportion_Homo']:.2f} & "
            f"{row['Mean KS stat_SV']:.4f} & "
            f"{row['Acceptance Proportion_SV']:.2f} \\\\"
        )

    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(rf"\caption{{{caption}}}")
    print(rf"\label{{{label}}}")
    print(r"\end{table}")


print_latex_combined_table(
    table1_near_stationary,
    caption=(
        r"KS diagnostics for the nearly stationary statistic $T_n$. "
        r"Higher acceptance proportions indicate closer agreement with the Gaussian limit $N(0,2c)$."
    ),
    label="tab:near_stationary_ks"
)

print_latex_combined_table(
    table2_mild_explosive,
    caption=(
        r"KS diagnostics for the mildly explosive statistic $S_n$. "
        r"Higher acceptance proportions indicate closer agreement with the standard Cauchy limit."
    ),
    label="tab:mild_explosive_ks"
)